# 🚗 Toronto Traffic Collision Severity Prediction
## Notebook 1 of 5 — Data Collection & Preparation

**Author:** Nishi Bhavesh Patel | Student ID: 501356244  
**Program:** Master of Data Science & Analytics  

---
### What this notebook does:
- Loads the Toronto collision CSV
- Fixes the Unix timestamp date column
- Fixes GPS coordinates
- Downloads hourly weather data from Government of Canada
- Cleans weather data using numerical columns (precipitation, temperature)
- Merges collision + weather on date and hour
- Saves final merged file: `collisions_with_weather.csv`

### Output file for next notebook: `collisions_with_weather.csv`

---

## Step 1 — Import Libraries

In [32]:
# !pip install pandas numpy requests pytz

import numpy as np
import pandas as pd
import requests
import os
import time
import warnings
from io import StringIO
warnings.filterwarnings('ignore')

COLLISION_FILE = '../MRP - Final Sem/Data/toronto_collisions.csv'
WEATHER_FILE   = '../MRP - Final Sem/Data/toronto_weather_hourly.csv'
MERGED_FILE    = '../MRP - Final Sem/Data/collisions_with_weather.csv'

STATION_ID   = 31688
RANDOM_STATE = 42

print('Libraries loaded!')

Libraries loaded!


---
## Step 2 — Load Collision Data

In [33]:
collisions = pd.read_csv(COLLISION_FILE, low_memory=False)
print(f'Raw shape: {collisions.shape}')
collisions.head(3)

Raw shape: (809034, 21)


,_id,OCC_DATE,OCC_MONTH,OCC_DOW,OCC_YEAR,OCC_HOUR,DIVISION,FATALITIES,INJURY_COLLISIONS,FTR_COLLISIONS,...,HOOD_158,NEIGHBOURHOOD_158,LONG_WGS84,LAT_WGS84,AUTOMOBILE,MOTORCYCLE,PASSENGER,BICYCLE,PEDESTRIAN,geometry
0,1,1.388550e+12,January,Wednesday,2014,1,D52,NaN,NO,NO,...,168,Downtown Yonge East (168),-79.378428,43.65041,YES,NO,NO,NO,NO,"{""coordinates"": [[-79.378427745, 43.65041009]]..."
1,2,1.388550e+12,January,Wednesday,2014,22,D32,NaN,YES,NO,...,NSA,NSA,0.000000,0.00000,YES,NO,NO,NO,NO,"{""coordinates"": [[5.6843418860808e-14, 5.08888..."
2,3,1.388550e+12,January,Wednesday,2014,2,NSA,NaN,YES,NO,...,NSA,NSA,0.000000,0.00000,YES,NO,NO,NO,NO,"{""coordinates"": [[5.6843418860808e-14, 5.08888..."


In [34]:
print('Missing values (top 10):')
print(collisions.isnull().sum().sort_values(ascending=False).head(10))

Missing values (top 10):
FATALITIES           808368
FTR_COLLISIONS            4
PD_COLLISIONS             4
INJURY_COLLISIONS         4
PASSENGER                 4
PEDESTRIAN                4
BICYCLE                   4
AUTOMOBILE                4
MOTORCYCLE                4
_id                       0
dtype: int64


---
## Step 3 — Fix Dates & Remove Duplicates

In [35]:
# Remove duplicates
before = len(collisions)
collisions = collisions.drop_duplicates(subset=['_id'])
print(f'Removed {before - len(collisions):,} duplicate rows')
print(f'Remaining: {len(collisions):,}')

Removed 0 duplicate rows
Remaining: 809,034


In [36]:
# Fix OCC_DATE — Unix timestamp in milliseconds → Toronto local time
collisions['datetime'] = (
    pd.to_datetime(collisions['OCC_DATE'], unit='ms', errors='coerce')
    .dt.tz_localize('UTC')
    .dt.tz_convert('America/Toronto')
    .dt.tz_localize(None)
    .dt.floor('H')
)
collisions = collisions.dropna(subset=['datetime'])
print(f'datetime range: {collisions["datetime"].min()} to {collisions["datetime"].max()}')

datetime range: 2013-12-31 23:00:00 to 2026-03-31 00:00:00


In [38]:
# Fix OCC_MONTH from string to number
month_map = {
    'January':1,'February':2,'March':3,'April':4,
    'May':5,'June':6,'July':7,'August':8,
    'September':9,'October':10,'November':11,'December':12
}
collisions['OCC_MONTH'] = collisions['OCC_MONTH'].map(month_map)
collisions['OCC_HOUR']  = collisions['datetime'].dt.hour

print('Time columns:')
print(f'  OCC_YEAR  : {collisions["OCC_YEAR"].min()} to {collisions["OCC_YEAR"].max()}')
print(f'  OCC_MONTH : {sorted(collisions["OCC_MONTH"].dropna().unique().tolist())}')
print(f'  OCC_DOW   : {sorted(collisions["OCC_DOW"].unique().tolist())}')
print(f'  OCC_HOUR  : {collisions["OCC_HOUR"].min()} to {collisions["OCC_HOUR"].max()}')

Time columns:
  OCC_YEAR  : 2014 to 2026
  OCC_MONTH : []
  OCC_DOW   : ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
  OCC_HOUR  : 0 to 23


---
## Step 4 — Fix GPS Coordinates

In [39]:
collisions['LAT_WGS84']  = collisions['LAT_WGS84'].replace(0.0, np.nan)
collisions['LONG_WGS84'] = collisions['LONG_WGS84'].replace(0.0, np.nan)

print(f'Valid GPS:   {collisions["LAT_WGS84"].notna().sum():,}')
print(f'Invalid GPS: {collisions["LAT_WGS84"].isna().sum():,} (will be dropped in Notebook 2)')

Valid GPS:   677,056
Invalid GPS: 131,978 (will be dropped in Notebook 2)


In [40]:
print('='*55)
print('  COLLISION DATA OVERVIEW')
print('='*55)
print(f'  Rows:     {len(collisions):,}')
print(f'  Columns:  {collisions.shape[1]}')
print(f'\n  Year distribution:')
print(collisions['OCC_YEAR'].value_counts().sort_index().to_string())
print(f'\n  Fatalities > 0 : {(collisions["FATALITIES"] > 0).sum():,}')
print(f'  Injury = YES   : {(collisions["INJURY_COLLISIONS"]=="YES").sum():,}')
print(f'  Minor (rest)   : {((collisions["FATALITIES"].fillna(0)==0) & (collisions["INJURY_COLLISIONS"]!="YES")).sum():,}')
print('='*55)

  COLLISION DATA OVERVIEW
  Rows:     809,034
  Columns:  22

  Year distribution:
OCC_YEAR
2014    64596
2015    67265
2016    69669
2017    74209
2018    79271
2019    82832
2020    44738
2021    43745
2022    59173
2023    67542
2024    70181
2025    67542
2026    18271

  Fatalities > 0 : 666
  Injury = YES   : 109,715
  Minor (rest)   : 698,656


---
## Step 5 — Download Weather Data

**Station:** Toronto City (ID: 31688) — downloads only months with collisions.  
Loads from disk if already downloaded.

In [41]:
def download_month(station_id, year, month):
    url = (
        f"https://climate.weather.gc.ca/climate_data/bulk_data_e.html?"
        f"format=csv&stationID={station_id}"
        f"&Year={year}&Month={month}"
        f"&Day=1&timeframe=1&submit=Download+Data"
    )
    try:
        response = requests.get(url, timeout=30)
        raw_text = response.text
        lines    = raw_text.split('\n')
        header_row = 0
        for i, line in enumerate(lines):
            if 'Date/Time' in line:
                header_row = i
                break
        df = pd.read_csv(StringIO(raw_text), skiprows=header_row)
        df = df[df['Date/Time (LST)'].notna()]
        df = df[df['Date/Time (LST)'].astype(str).str.startswith(str(year))]
        return df
    except Exception as e:
        print(f'  ERROR {year}-{month:02d}: {e}')
        return None

if os.path.exists(WEATHER_FILE):
    print(f'Loading existing: {WEATHER_FILE}')
    weather_raw = pd.read_csv(WEATHER_FILE, low_memory=False)
    print(f'Loaded {len(weather_raw):,} rows')
else:
    unique_months = sorted(collisions['datetime'].dt.to_period('M').unique())
    print(f'Downloading {len(unique_months)} months from station {STATION_ID}...')
    all_frames = []
    for i, period in enumerate(unique_months):
        year  = period.year
        month = period.month
        print(f'[{i+1}/{len(unique_months)}] {year}-{month:02d}...', end=' ')
        df_m = download_month(STATION_ID, year, month)
        if df_m is not None and len(df_m) > 0:
            all_frames.append(df_m)
            print(f'OK ({len(df_m)} rows)')
        else:
            print('SKIPPED')
        time.sleep(0.5)
    weather_raw = pd.concat(all_frames, ignore_index=True)
    weather_raw.to_csv(WEATHER_FILE, index=False)
    print(f'\nSaved {len(weather_raw):,} rows')

Loading existing: ../MRP - Final Sem/Data/toronto_weather_hourly.csv
Loaded 108,096 rows


---
## Step 6 — Clean Weather Data

Using **numerical columns** (precipitation, temperature, humidity, dew_point) to derive weather flags.  
Note: Wind speed and Visibility are not recorded at this station.

In [42]:
# Check available columns
print('Non-null counts:')
for col in ['Temp (\u00b0C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)',
            'Visibility (km)', 'Dew Point Temp (\u00b0C)', 'Rel Hum (%)']:
    if col in weather_raw.columns:
        print(f'  {col}: {weather_raw[col].notna().sum():,} non-null')

Non-null counts:
  Temp (°C): 107,719 non-null
  Precip. Amount (mm): 106,950 non-null
  Wind Spd (km/h): 0 non-null
  Visibility (km): 0 non-null
  Dew Point Temp (°C): 107,721 non-null
  Rel Hum (%): 107,721 non-null


In [43]:
# Select and rename columns — Precip. Amount instead of text Weather column
weather = weather_raw[[
    'Date/Time (LST)',
    'Temp (\u00b0C)',
    'Rel Hum (%)',
    'Dew Point Temp (\u00b0C)',
    'Precip. Amount (mm)'
]].copy()

weather.columns = ['datetime', 'temperature', 'humidity', 'dew_point', 'precipitation']

# Convert to numeric
for col in ['temperature', 'humidity', 'dew_point', 'precipitation']:
    weather[col] = pd.to_numeric(weather[col], errors='coerce')

# Parse datetime
weather['datetime'] = pd.to_datetime(weather['datetime'], errors='coerce').dt.floor('H')

# Fill missing values
for col in ['temperature', 'humidity', 'dew_point']:
    weather[col] = weather[col].fillna(method='ffill').fillna(method='bfill')
weather['precipitation'] = weather['precipitation'].fillna(0)

# Remove duplicates and sort
weather = weather.drop_duplicates(subset='datetime')
weather = weather.dropna(subset=['datetime'])
weather = weather.sort_values('datetime').reset_index(drop=True)

print(f'Weather cleaned: {len(weather):,} rows')
print(f'Range: {weather["datetime"].min()} to {weather["datetime"].max()}')
weather.head(3)

Weather cleaned: 108,096 rows
Range: 2013-12-01 00:00:00 to 2026-03-31 23:00:00


,datetime,temperature,humidity,dew_point,precipitation
0,2013-12-01 00:00:00,3.3,70.0,-1.6,0.0
1,2013-12-01 01:00:00,3.6,71.0,-1.2,0.0
2,2013-12-01 02:00:00,3.6,72.0,-1.0,0.0


In [44]:
# Derive weather flags from numerical values
weather['is_rain'] = (weather['precipitation'] > 0.2).astype(int)
weather['is_snow'] = ((weather['precipitation'] > 0.2) & (weather['temperature'] <= 2.0)).astype(int)
weather['is_ice']  = ((weather['temperature'] <= 0.0) & (weather['precipitation'] > 0.2)).astype(int)
weather['adverse_weather_w'] = (
    (weather['is_rain'] == 1) |
    (weather['is_snow'] == 1) |
    (weather['is_ice']  == 1)
).astype(int)

print('Weather flags:')
for col in ['is_rain', 'is_snow', 'is_ice', 'adverse_weather_w']:
    pct = weather[col].mean() * 100
    print(f'  {col:20s}: {weather[col].sum():,} hours ({pct:.1f}%)')

Weather flags:
  is_rain             : 6,346 hours (5.9%)
  is_snow             : 1,858 hours (1.7%)
  is_ice              : 1,124 hours (1.0%)
  adverse_weather_w   : 6,346 hours (5.9%)


---
## Step 7 — Merge Collision + Weather

In [50]:
# Left join — every collision gets its hour's weather
df = collisions.merge(weather, on='datetime', how='left')

matched = df['temperature'].notna().sum()
print(f'Match rate: {matched:,} / {len(df):,} ({matched/len(df)*100:.1f}%)')
print(f'Merged shape: {df.shape}')
df.head(3)

Match rate: 809,034 / 809,034 (100.0%)
Merged shape: (809034, 30)


,_id,OCC_DATE,OCC_MONTH,OCC_DOW,OCC_YEAR,OCC_HOUR,DIVISION,FATALITIES,INJURY_COLLISIONS,FTR_COLLISIONS,...,geometry,datetime,temperature,humidity,dew_point,precipitation,is_rain,is_snow,is_ice,adverse_weather_w
0,1,1.388550e+12,NaN,Wednesday,2014,23,D52,NaN,NO,NO,...,"{""coordinates"": [[-79.378427745, 43.65041009]]...",2013-12-31 23:00:00,-8.8,81.0,-11.5,0.0,0,0,0,0
1,2,1.388550e+12,NaN,Wednesday,2014,23,D32,NaN,YES,NO,...,"{""coordinates"": [[5.6843418860808e-14, 5.08888...",2013-12-31 23:00:00,-8.8,81.0,-11.5,0.0,0,0,0,0
2,3,1.388550e+12,NaN,Wednesday,2014,23,NSA,NaN,YES,NO,...,"{""coordinates"": [[5.6843418860808e-14, 5.08888...",2013-12-31 23:00:00,-8.8,81.0,-11.5,0.0,0,0,0,0


In [51]:
# Fix OCC_MONTH — derive directly from datetime (most reliable)
df['OCC_MONTH'] = df['datetime'].dt.month

print(f'OCC_MONTH sample: {df["OCC_MONTH"].head(5).tolist()}')
print(f'OCC_MONTH unique: {sorted(df["OCC_MONTH"].dropna().unique().tolist())}')
print(f'OCC_MONTH NaN:    {df["OCC_MONTH"].isna().sum():,}')

OCC_MONTH sample: [12, 12, 12, 12, 12]
OCC_MONTH unique: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
OCC_MONTH NaN:    0


In [52]:
# Save merged file
df.to_csv(MERGED_FILE, index=False)
print(f'Saved: {MERGED_FILE}')
print(f'Shape: {df.shape}')
print(f'\nFinal columns: {df.columns.tolist()}')
print('\n✅ Notebook 1 complete! Open Notebook 2 next.')

Saved: ../MRP - Final Sem/Data/collisions_with_weather.csv
Shape: (809034, 30)

Final columns: ['_id', 'OCC_DATE', 'OCC_MONTH', 'OCC_DOW', 'OCC_YEAR', 'OCC_HOUR', 'DIVISION', 'FATALITIES', 'INJURY_COLLISIONS', 'FTR_COLLISIONS', 'PD_COLLISIONS', 'HOOD_158', 'NEIGHBOURHOOD_158', 'LONG_WGS84', 'LAT_WGS84', 'AUTOMOBILE', 'MOTORCYCLE', 'PASSENGER', 'BICYCLE', 'PEDESTRIAN', 'geometry', 'datetime', 'temperature', 'humidity', 'dew_point', 'precipitation', 'is_rain', 'is_snow', 'is_ice', 'adverse_weather_w']

✅ Notebook 1 complete! Open Notebook 2 next.
